<a href="https://colab.research.google.com/github/JxseDiaz/Deep-Learning/blob/main/Parcial_1_Deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción de churn con un MLP – Telco Customer Churn
**DLY0100 Deep Learning – Evaluación Parcial N°1**
Integrantes: *Victoria Castillo – José Diaz*

## 1. Introducción
Queremos predecir si un cliente de una empresa de telecomunicaciones va a abandonar el servicio (churn) usando sus datos de contrato, servicios y cobros. Es un problema de clasificación binaria y lo resolvemos con una red neuronal multicapa (MLP) en Keras.

Solo cerca del 27% de los clientes abandona, así que mirar solo la accuracy no sirve (un modelo que siempre diga "no abandona" ya tendría ~73%). Por eso también usamos precision, recall y F1. Para la empresa es peor no detectar a un cliente que se va (falso negativo) que ofrecerle un descuento a uno que se iba a quedar, así que el recall es importante.

In [ ]:
import os, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
EPOCHS = 30          # épocas de los experimentos
EPOCHS_LARGO = 100   # para ver el efecto del número de épocas
keras.utils.set_random_seed(SEED)

## 2. Carga y preprocesamiento de datos

In [ ]:
archivo = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

if os.path.exists(archivo):
    df = pd.read_csv(archivo)
else:
    try:
        df = pd.read_csv(url)
    except Exception:
        from google.colab import files        # si falla la descarga, subir el CSV de Kaggle a mano
        subido = files.upload()
        df = pd.read_csv(io.BytesIO(next(iter(subido.values()))))

print(df.shape)
df.head()

In [ ]:
print(df["Churn"].value_counts(normalize=True).round(3))
df["Churn"].value_counts().plot(kind="bar", color=["steelblue", "indianred"], rot=0, title="Distribución de Churn")
plt.show()

Decisiones de preprocesamiento:
- Se elimina `customerID`, porque es solo un identificador y no sirve para predecir.
- `TotalCharges` viene como texto porque algunas filas tienen un espacio en blanco. La convertimos a número y esos 11 casos se rellenan con 0 (se comprueba abajo que son clientes con `tenure = 0`, o sea que todavía no se les cobra nada).
- "No internet service" y "No phone service" se cambian por "No", porque ya están representados en `InternetService` y `PhoneService`.
- `Churn` se pasa a 1/0.
- Las variables categóricas se transforman con one-hot encoding, porque la red solo trabaja con números y así no se inventa un orden entre categorías.
- Se divide en train/validación/test (70/15/15) con `stratify` para mantener el ~27% de churn en los tres conjuntos. El test se usa una sola vez al final.
- Se escala con `StandardScaler` ajustado **solo con train**, para no filtrar información de validación/test. Se escala porque variables como `TotalCharges` (miles) y las dummies (0/1) tienen escalas muy distintas y eso dificulta el entrenamiento.

In [ ]:
datos = df.drop(columns="customerID")
datos["TotalCharges"] = pd.to_numeric(datos["TotalCharges"], errors="coerce")
nulos = datos["TotalCharges"].isna()
print("Nulos en TotalCharges:", nulos.sum(), "| todos con tenure=0:", (datos.loc[nulos, "tenure"] == 0).all())
datos["TotalCharges"] = datos["TotalCharges"].fillna(0)

datos = datos.replace({"No internet service": "No", "No phone service": "No"})
datos["Churn"] = (datos["Churn"] == "Yes").astype(int)
datos = pd.get_dummies(datos, drop_first=True, dtype=float)

X = datos.drop(columns="Churn").values.astype("float32")
y = datos["Churn"].values
print("Variables de entrada:", X.shape[1])

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

scaler = StandardScaler().fit(X_tr)
X_tr, X_val, X_te = scaler.transform(X_tr), scaler.transform(X_val), scaler.transform(X_te)
print("train:", X_tr.shape, "| val:", X_val.shape, "| test:", X_te.shape)

## 3. Funciones de activación, error y salida
Programamos a mano las funciones que usa la red:
- **Activación (capas ocultas):** ReLU, tanh y sigmoide. Sin una función no lineal, varias capas equivaldrían a una sola transformación lineal.
- **Salida:** sigmoide con una neurona, que entrega la probabilidad de que el cliente abandone (entre 0 y 1).
- **Error:** entropía cruzada binaria (BCE), que es la que corresponde a una salida sigmoide en clasificación binaria. La comparamos con el error cuadrático medio (MSE).

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)
def tanh(z):    return np.tanh(z)

def bce(y, p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def mse(y, p):
    return np.mean((y - p) ** 2)

# revisamos que den lo mismo que Keras
z = np.linspace(-6, 6, 200)
for nombre, f in [("sigmoid", sigmoid), ("relu", relu), ("tanh", tanh)]:
    print(nombre, np.allclose(f(z), keras.activations.get(nombre)(tf.constant(z, dtype=tf.float32)).numpy(), atol=1e-5))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for nombre, f in [("sigmoid", sigmoid), ("relu", relu), ("tanh", tanh)]:
    ax[0].plot(z, f(z), label=nombre)
ax[0].set_title("Funciones de activación"); ax[0].legend()
p = np.linspace(0.001, 0.999, 200)
ax[1].plot(p, -np.log(p), label="BCE"); ax[1].plot(p, (1 - p) ** 2, label="MSE")
ax[1].set_title("Error cuando la clase real es 1"); ax[1].set_xlabel("probabilidad predicha"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Modelo MLP y modelo base
Usamos un MLP porque los datos son tabulares. La función `crear_modelo` permite cambiar un hiperparámetro a la vez en los experimentos. Cada bloque oculto es Dense → (BatchNorm) → activación → (Dropout).

Modelo base: 2 capas ocultas (64 y 32 neuronas), ReLU, salida sigmoide, BCE, Adam con lr = 0.001, batch de 32 y 30 épocas. Son valores típicos para partir; después los vamos ajustando.

In [ ]:
def crear_modelo(n_in, capas=(64, 32), activacion="relu", dropout=0.0, l2=0.0,
                 batchnorm=False, lr=0.001, loss="binary_crossentropy"):
    modelo = keras.Sequential([layers.Input(shape=(n_in,))])
    reg = keras.regularizers.l2(l2) if l2 > 0 else None
    for n in capas:
        modelo.add(layers.Dense(n, kernel_regularizer=reg))
        if batchnorm:
            modelo.add(layers.BatchNormalization())
        modelo.add(layers.Activation(activacion))
        if dropout > 0:
            modelo.add(layers.Dropout(dropout))
    modelo.add(layers.Dense(1, activation="sigmoid"))
    modelo.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=["accuracy"])
    return modelo

resultados = []   # acá guardamos las métricas (en validación) de cada prueba

def metricas(y_real, prob, umbral=0.5):
    pred = (prob >= umbral).astype(int)
    return {"accuracy": accuracy_score(y_real, pred), "precision": precision_score(y_real, pred, zero_division=0),
            "recall": recall_score(y_real, pred, zero_division=0), "f1": f1_score(y_real, pred, zero_division=0)}

def entrenar(nombre, grupo, epochs=EPOCHS, batch_size=32, early_stop=False, class_weight=None, **kwargs):
    keras.utils.set_random_seed(SEED)       # misma semilla siempre, para comparar justo
    modelo = crear_modelo(X_tr.shape[1], **kwargs)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)] if early_stop else []
    h = modelo.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size,
                   class_weight=class_weight, callbacks=cb, verbose=0).history
    prob = modelo.predict(X_val, verbose=0).ravel()
    fila = {"grupo": grupo, "config": nombre, **metricas(y_val, prob),
            "loss_train": h["loss"][-1], "loss_val": h["val_loss"][-1], "epocas": len(h["loss"])}
    resultados.append(fila)
    return modelo, h

def graficar(historiales, titulo):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for i, (nombre, h) in enumerate(historiales.items()):
        c = plt.cm.tab10(i)
        ax[0].plot(h["loss"], "--", color=c, alpha=0.6)
        ax[0].plot(h["val_loss"], color=c, label=nombre)
        ax[1].plot(h["val_accuracy"], color=c, label=nombre)
    ax[0].set_title(titulo + " – loss (punteada = train, continua = validación)")
    ax[1].set_title("Accuracy en validación")
    ax[0].set_xlabel("época"); ax[1].set_xlabel("época"); ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

def tabla(grupo):
    t = pd.DataFrame(resultados)
    return t[t["grupo"] == grupo].set_index("config")[["accuracy", "precision", "recall", "f1", "loss_train", "loss_val", "epocas"]].round(3)

In [ ]:
modelo_base, h_base = entrenar("base", "base")
graficar({"base": h_base}, "Modelo base")
tabla("base")

**Análisis del modelo base:** *(escribir acá: ¿converge?, ¿en qué época empieza a separarse la loss de validación de la de train?, ¿qué tal el recall comparado con la accuracy?)*

## 5. Experimentos controlados
En cada experimento cambiamos **un solo parámetro** y dejamos todo lo demás igual que el modelo base (misma semilla, mismos datos).

### 5.1 Tasa de aprendizaje

In [ ]:
hist = {}
for lr in [0.0001, 0.001, 0.01, 0.1]:
    _, hist[f"lr={lr}"] = entrenar(f"lr={lr}", "lr", lr=lr)
graficar(hist, "Tasa de aprendizaje")
tabla("lr")

**Análisis:** *(escribir acá qué lr converge más rápido, cuál es inestable y cuál elegimos, con números de la tabla)*

### 5.2 Tamaño de batch

In [ ]:
hist = {}
for bs in [16, 32, 64, 128, 256]:
    _, hist[f"batch={bs}"] = entrenar(f"batch={bs}", "batch", batch_size=bs)
graficar(hist, "Tamaño de batch")
tabla("batch")

**Análisis:** *(escribir acá)*

### 5.3 Número de épocas
Entrenamos una vez con 100 épocas y miramos en qué momento deja de mejorar la validación.

In [ ]:
_, h_largo = entrenar("100 épocas", "épocas", epochs=EPOCHS_LARGO)
mejor = int(np.argmin(h_largo["val_loss"])) + 1
plt.plot(h_largo["loss"], "--", label="train"); plt.plot(h_largo["val_loss"], label="validación")
plt.axvline(mejor - 1, color="red", linestyle=":", label=f"mínimo val_loss (época {mejor})")
plt.xlabel("época"); plt.ylabel("loss"); plt.legend(); plt.title("Efecto del número de épocas"); plt.show()

**Análisis:** *(escribir acá: qué pasa después de la época del mínimo y qué implica para elegir las épocas)*

### 5.4 Número de capas y neuronas

In [ ]:
hist = {}
for nombre, capas in [("1 capa (32)", (32,)), ("2 capas (64-32)", (64, 32)), ("3 capas (128-64-32)", (128, 64, 32))]:
    _, hist[nombre] = entrenar(nombre, "capas", capas=capas)
graficar(hist, "Arquitectura")
tabla("capas")

**Análisis:** *(escribir acá)*

## 6. Comparación de funciones de activación y de error
Comparamos las activaciones de las capas ocultas y las funciones de error. Como la loss de BCE y la de MSE tienen escalas distintas, la comparación justa es con las métricas de la tabla.

In [ ]:
hist = {}
for act in ["relu", "tanh", "sigmoid"]:
    _, hist[act] = entrenar(act, "activación", activacion=act)
graficar(hist, "Función de activación")
tabla("activación")

In [ ]:
hist = {}
for nombre, loss in [("BCE", "binary_crossentropy"), ("MSE", "mean_squared_error")]:
    _, hist[nombre] = entrenar(nombre, "error", loss=loss)
graficar(hist, "Función de error")
tabla("error")

**Análisis y elección:** *(escribir acá qué activación y qué función de error elegimos y por qué, usando las tablas. Relacionarlo con lo visto en la sección 3: derivada de la sigmoide, penalización de BCE vs MSE.)*

## 7. Regularización y optimización
Para reducir el sobreajuste probamos dropout (apaga neuronas al azar en cada paso), regularización L2 (penaliza pesos grandes), batch normalization (normaliza las activaciones de cada capa) y early stopping (corta el entrenamiento cuando la validación deja de mejorar).

In [ ]:
hist = {}
pruebas = [("sin regularización", {}),
           ("dropout 0.3", {"dropout": 0.3}),
           ("L2 0.001", {"l2": 0.001}),
           ("batchnorm", {"batchnorm": True}),
           ("early stopping", {"early_stop": True, "epochs": EPOCHS_LARGO})]
for nombre, kw in pruebas:
    _, hist[nombre] = entrenar(nombre, "regularización", **kw)
graficar(hist, "Regularización")
t = tabla("regularización")
t["brecha"] = (t["loss_val"] - t["loss_train"]).round(3)     # val - train: más grande = más sobreajuste
t

**Análisis:** *(escribir acá qué técnica reduce más el sobreajuste, si mejora las métricas o solo suaviza las curvas, y cómo cambia la estabilidad)*

## 8. Ajuste de hiperparámetros y modelo final
Con lo visto en los experimentos armamos la configuración final. Como el recall era bajo en el modelo base, probamos también `class_weight`, que le da más peso al error en la clase minoritaria (los que abandonan).

*(Cambiar los valores de `config_final` según lo que haya salido mejor en las tablas anteriores y justificarlo abajo.)*

In [ ]:
config_final = dict(capas=(64, 32), activacion="relu", loss="binary_crossentropy",
                    dropout=0.3, lr=0.001, batch_size=64, early_stop=True, epochs=EPOCHS_LARGO)

pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr)
pesos = {0: pesos[0], 1: pesos[1]}

modelo_sin, h_sin = entrenar("final sin pesos", "final", **config_final)
modelo_con, h_con = entrenar("final con class_weight", "final", class_weight=pesos, **config_final)
graficar({"sin pesos": h_sin, "con class_weight": h_con}, "Modelo final")

base = tabla("base").rename(index={"base": "modelo base"})
pd.concat([base, tabla("final")])

**Justificación de la configuración final:** *(escribir acá por qué elegimos cada valor, citando las tablas, y qué pasó con precision y recall al usar class_weight)*

## 9. Evaluación final (conjunto de test)
Acá se usa el test por primera y única vez.

In [ ]:
modelo_final = modelo_con        # cambiar a modelo_sin si decidimos no usar class_weight
p_base = modelo_base.predict(X_te, verbose=0).ravel()
p_final = modelo_final.predict(X_te, verbose=0).ravel()

resumen = pd.DataFrame({"modelo base": metricas(y_te, p_base), "modelo final": metricas(y_te, p_final)}).T.round(3)
print(resumen)
print()
print(classification_report(y_te, (p_final >= 0.5).astype(int), target_names=["No abandona", "Abandona"], digits=3))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for a, p, nombre in [(ax[0], p_base, "modelo base"), (ax[1], p_final, "modelo final")]:
    ConfusionMatrixDisplay(confusion_matrix(y_te, (p >= 0.5).astype(int)),
                           display_labels=["No abandona", "Abandona"]).plot(ax=a, colorbar=False, cmap="Blues")
    a.set_title(nombre); a.grid(False)
plt.tight_layout(); plt.show()

Cálculo de las métricas a mano con la matriz de confusión del modelo final:

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_te, (p_final >= 0.5).astype(int)).ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print("accuracy  =", round((tp + tn) / (tp + tn + fp + fn), 3))
print("precision =", round(tp / (tp + fp), 3), " (de los que predijo que se van, cuántos realmente se iban)")
print("recall    =", round(tp / (tp + fn), 3), " (de los que realmente se iban, cuántos detectó)")
pr, rc = tp / (tp + fp), tp / (tp + fn)
print("f1        =", round(2 * pr * rc / (pr + rc), 3))

**Interpretación:** *(escribir acá: qué significan los números para el negocio, cuántos clientes que se iban quedan sin detectar (FN), cuántas ofertas se "gastan" de más (FP), y cómo se podría mejorar precision y recall)*

## 10. Conclusiones
*(escribir acá, con los resultados reales: desempeño final del modelo, qué ajustes lo mejoraron más y cuánto respecto del modelo base, limitaciones del trabajo y posibles mejoras)*

## Referencias
- Documentación de TensorFlow / Keras: https://keras.io/api/
- Documentación de scikit-learn: https://scikit-learn.org/stable/
- Telco Customer Churn (Kaggle): https://www.kaggle.com/datasets/blastchar/telco-customer-churn